# 🚗 차량 손상 YOLO 학습 — Google Colab (GPU)

로컬 M1(MPS)보다 5~10배 빠른 무료 GPU(T4)로 학습합니다.

## 준비 (최초 1회)
1. 로컬에서 데이터셋 압축: `cd ~/Desktop/car-damage-analyzer/data && zip -r yolo_dataset.zip yolo_dataset`
2. `yolo_dataset.zip` 을 구글드라이브 `MyDrive/` 에 업로드
3. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
4. 이 노트북을 위에서부터 순서대로 실행

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 라이브러리 설치 + 코드 클론

In [ ]:
!pip install ultralytics -q

import os

# 이미 클론된 경우 다시 클론하지 않음
if not os.path.exists('/content/CarCheck'):
    !git clone https://github.com/EunSeok-222/CarCheck.git

# 절대경로로 이동 (중복 실행해도 안전)
%cd /content/CarCheck

## 3. 구글드라이브 마운트 + 데이터셋 압축해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 드라이브의 zip → /content/CarCheck/data/ 로 압축해제 (절대경로)
import os
if not os.path.exists('/content/CarCheck/data/yolo_dataset'):
    !unzip -q /content/drive/MyDrive/yolo_dataset.zip -d /content/CarCheck/data/
    print('✅ 압축해제 완료')
else:
    print('✅ 이미 압축해제됨, 스킵')

!ls /content/CarCheck/data/yolo_dataset/images/train | wc -l
!ls /content/CarCheck/data/yolo_dataset/images/val | wc -l

## 4. Colab 경로용 yaml 생성

In [ ]:
yaml_text = '''path: /content/CarCheck/data/yolo_dataset
train: images/train
val:   images/val
nc: 4
names:
  0: Scratched
  1: Breakage
  2: Separated
  3: Crushed
'''
with open('data/car_damage_colab.yaml', 'w') as f:
    f.write(yaml_text)
print('✅ yaml 생성 완료')

## 5. 파인튜닝 (GPU) — 중단/재시작 자동 복구

- **처음 실행**: 드라이브의 `best.pt` 기반으로 파인튜닝 시작
- **재시작 시**: 드라이브 `carcheck_checkpoint/last.pt` 발견 시 자동으로 이어서 학습
- **매 epoch**: `last.pt` + `args.yaml` → 드라이브 `carcheck_checkpoint/` 백업

In [ ]:
from ultralytics import YOLO
import shutil, os

DRIVE_CKPT  = '/content/drive/MyDrive/carcheck_checkpoint'
DRIVE_BEST  = '/content/drive/MyDrive/best.pt'
LOCAL_SAVE  = '/content/CarCheck/runs/carcheck_finetune'

os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(f'{LOCAL_SAVE}/weights', exist_ok=True)

def save_checkpoint(trainer):
    """매 epoch마다 last.pt + args.yaml → 드라이브 백업."""
    sd = str(trainer.save_dir)
    for fname, dst_name in [('weights/last.pt', 'last.pt'), ('args.yaml', 'args.yaml')]:
        src = f'{sd}/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{DRIVE_CKPT}/{dst_name}')
    if trainer.epoch % 10 == 0 and os.path.exists(str(trainer.best)):
        shutil.copy(str(trainer.best), f'{DRIVE_CKPT}/best.pt')
    print(f'  💾 epoch {trainer.epoch + 1}: 체크포인트 저장')

drive_last = f'{DRIVE_CKPT}/last.pt'
drive_args = f'{DRIVE_CKPT}/args.yaml'

if os.path.exists(drive_last):
    print('✅ 이전 체크포인트 발견 → 이어서 학습')
    shutil.copy(drive_last, f'{LOCAL_SAVE}/weights/last.pt')
    if os.path.exists(drive_args):
        shutil.copy(drive_args, f'{LOCAL_SAVE}/args.yaml')
    model = YOLO(f'{LOCAL_SAVE}/weights/last.pt')
    model.add_callback('on_train_epoch_end', save_checkpoint)
    results = model.train(resume=True)
else:
    print('🆕 파인튜닝 시작 (best.pt 기반, epochs=100)')
    shutil.copy(DRIVE_BEST, '/content/CarCheck/finetune_base.pt')
    model = YOLO('/content/CarCheck/finetune_base.pt')
    model.add_callback('on_train_epoch_end', save_checkpoint)
    results = model.train(
        data='data/car_damage_colab.yaml',
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        patience=25,
        project='/content/CarCheck/runs',
        name='carcheck_finetune',
        exist_ok=True,
    )

print('🎉 학습 완료')
print('Best mAP50(Box): ', results.results_dict.get('metrics/mAP50(B)', 0))
print('Best mAP50(Mask):', results.results_dict.get('metrics/mAP50(M)', 0))

best_pt = f'{str(results.save_dir)}/weights/best.pt'
if os.path.exists(best_pt):
    shutil.copy(best_pt, DRIVE_BEST)
    print('✅ best.pt → 드라이브 최종 저장 완료')

## 6. 결과 그래프 + best.pt 드라이브에 저장

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# 실제 저장된 results.png 자동 탐색
result_pngs = glob.glob('/content/CarCheck/runs/segment/**/results.png', recursive=True)
if result_pngs:
    display(IPImage(result_pngs[0]))
else:
    print('results.png 없음')

In [ ]:
from google.colab import files
import glob

# best.pt 자동 탐색 후 다운로드
best_pts = glob.glob('/content/CarCheck/runs/segment/**/best.pt', recursive=True)
if best_pts:
    print(f'📁 best.pt 경로: {best_pts[0]}')
    files.download(best_pts[0])
else:
    print('❌ best.pt 없음 — 드라이브에서 확인하세요: MyDrive/best.pt')

## 학습 후 로컬 적용

드라이브/다운로드받은 `best.pt` 를 로컬 `car-damage-analyzer/models/best.pt` 에 덮어쓰면
Streamlit 앱이 자동으로 새 모델을 사용합니다.